# KENYA_SACCO_SIM Benchmark Walkthrough

This notebook is a lightweight audit path for the latest generated benchmark package. It uses only the Python standard library so it can run before optional analysis dependencies are installed.

In [ ]:
from pathlib import Path
import csv
import json

DATASET_DIR = Path('../datasets/KENYA_SACCO_SIM_v1_100k').resolve()
DATASET_DIR

## File Counts

Start by confirming the package has the expected files and validation status.

In [ ]:
def row_count(path):
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        return sum(1 for _ in reader) if reader.fieldnames else 0

manifest = json.loads((DATASET_DIR / 'manifest.json').read_text())
validation = json.loads((DATASET_DIR / 'validation_report.json').read_text())
counts = {name: row_count(DATASET_DIR / name) for name in manifest['files'] if name.endswith('.csv')}
print('validation:', len(validation['errors']), 'errors /', len(validation['warnings']), 'warnings')
counts

## Label Tables

`alerts_truth.csv` is detailed context. `pattern_labels.csv` is one row per suspicious case. `edge_labels.csv` is sparse graph-edge context for edge-supervised tasks.

In [ ]:
for label_file in ['alerts_truth.csv', 'pattern_labels.csv', 'edge_labels.csv']:
    path = DATASET_DIR / label_file
    print(label_file, row_count(path) if path.exists() else 'missing')

with (DATASET_DIR / 'pattern_labels.csv').open(newline='', encoding='utf-8') as handle:
    pattern_rows = list(csv.DictReader(handle))
print('unique patterns:', len({row['pattern_id'] for row in pattern_rows}))
print('typologies:', sorted({row['typology'] for row in pattern_rows}))

## Rule And Confounder Diagnostics

Read rule precision/recall together with confounder diagnostics. Rule-vs-ML comparison is descriptive, not a production claim.

In [ ]:
rule_results = json.loads((DATASET_DIR / 'rule_results.json').read_text())
confounders = json.loads((DATASET_DIR / 'benchmark_confounder_diagnostics.json').read_text())
for typology, section in sorted(rule_results.items()):
    if isinstance(section, dict) and 'precision' in section:
        print(typology, 'precision=', section['precision'], 'recall=', section['recall'], 'candidates=', section['candidate_count'])
print('confounder risk:', confounders.get('risk_summary'))

## Consumer Rules

- Do not train member-level models on label CSVs.
- Use `pattern_labels.csv` or `pattern_id` for unique case counts.
- Hold out or stratify static confounders such as `persona_type`, `member_type`, `dormant_flag`, `age`, and `devices.last_seen` before ML lift claims.
- Filter `SOURCE_ACCOUNT` and `SINK_ACCOUNT` from customer-account risk aggregation.